In [76]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 


In [77]:
df = pd.read_csv("qoute_dataset.csv")

In [78]:
df

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe
...,...,...
3033,The past beats inside me like a second heart.,"John Banville,"
3034,"Damn, Claire. Warn a guy before you do a face-...","Rachel Caine,"
3035,"Can you be a girl for a few seconds?""""I'm alwa...","Veronica Roth,"
3036,That's what fiction is for. It's for getting a...,Tim O'Brien


In [79]:
df['quote'][0]

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

In [80]:
df.shape

(3038, 2)

In [81]:
quotes = df['quote']
quotes.head()

0    “The world as we have created it is a process ...
1    “It is our choices, Harry, that show what we t...
2    “There are only two ways to live your life. On...
3    “The person, be it gentleman or lady, who has ...
4    “Imperfection is beauty, madness is genius and...
Name: quote, dtype: object

In [82]:
quotes = quotes.str.lower()

In [83]:
import string
translator = str.maketrans('','',string.punctuation) 
quotes = quotes.apply(lambda x:x.translate(translator)) 

In [84]:
quotes.head()

0    “the world as we have created it is a process ...
1    “it is our choices harry that show what we tru...
2    “there are only two ways to live your life one...
3    “the person be it gentleman or lady who has no...
4    “imperfection is beauty madness is genius and ...
Name: quote, dtype: object

In [85]:
from tensorflow.keras.preprocessing.text import Tokenizer # type: ignore

In [86]:
vocab_size = 10000

tokinizer = Tokenizer(num_words = vocab_size)
tokinizer.fit_on_texts(quotes) 

In [87]:
word_index = tokinizer.word_index
print(len(word_index))
list(word_index.items())[:10] 

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [88]:
sequence = tokinizer.texts_to_sequences(quotes)

In [89]:
for i in range(3):
    print(quotes[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [90]:
for i in range(3):
    print(sequence[i])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


In [91]:
X = []
y = []
for seq in sequence:
    for i in range(1,len(seq)):
        input_seq = seq[:i]
        output_seq = seq[i]
        X.append(input_seq)
        y.append(output_seq)

In [92]:
len(X)

85271

In [93]:
len(y)

85271

In [94]:
max_len = max(len(x) for x in X)
print(max_len)

745


In [95]:
from tensorflow.keras.preprocessing.sequence import pad_sequences # type: ignore
X_padded = pad_sequences(X, maxlen = max_len, padding='pre')

In [96]:
X_padded

array([[   0,    0,    0, ...,    0,    0,  713],
       [   0,    0,    0, ...,    0,  713,   62],
       [   0,    0,    0, ...,  713,   62,   29],
       ...,
       [   0,    0,    0, ...,    9,   19, 1125],
       [   0,    0,    0, ...,   19, 1125,    3],
       [   0,    0,    0, ..., 1125,    3,  169]],
      shape=(85271, 745), dtype=int32)

In [97]:
y = np.array(y)


In [98]:
X_padded.shape

(85271, 745)

In [99]:
from tensorflow.keras.utils import to_categorical 
y_one_hot = to_categorical(y, num_classes = vocab_size)

In [100]:
y.shape

(85271,)

In [101]:
y_one_hot.shape

(85271, 10000)

In [102]:
from tensorflow.keras.models import Sequential # type: ignore
from tensorflow.keras.layers import Embedding,SimpleRNN,LSTM,Dense # type: ignore

In [103]:
embedding_dim = 50
rnn_units = 128

In [104]:
rnn_model = Sequential()
rnn_model.add(
    Embedding(input_dim = vocab_size,output_dim = embedding_dim,
    input_length = max_len)
    )

rnn_model.add(SimpleRNN(units = rnn_units))
rnn_model.add(Dense(units = vocab_size,activation = 'softmax'))

c:\Users\Nemo96\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [105]:
rnn_model.compile(
    optimizer = 'adam',
    loss = 'categorical_crossentropy',
    metrics = ['accuracy']
)

In [106]:
rnn_model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [107]:
lstm_model = Sequential()
lstm_model.add(
    Embedding(input_dim = vocab_size,output_dim = embedding_dim,
    input_length = max_len)
)
lstm_model.add(LSTM(units = rnn_units))
lstm_model.add(Dense(units = vocab_size,activation = 'softmax'))

In [108]:
lstm_model.compile(
    optimizer = 'adam',
    loss = 'categorical_crossentropy',
    metrics = ['accuracy']
)

In [109]:
lstm_model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [110]:
epochs = 100
batch_size = 128 

In [111]:
history_rnn = rnn_model.fit(
    X_padded, y_one_hot,
    epochs = epochs,
    batch_size = batch_size,
    validation_split = 0.1
)

Epoch 1/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 80s 131ms/step - accuracy: 0.0456 - loss: 6.7062 - val_accuracy: 0.0590 - val_loss: 6.5411
Epoch 2/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 77s 128ms/step - accuracy: 0.0790 - loss: 6.1138 - val_accuracy: 0.0911 - val_loss: 6.3113
Epoch 3/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 77s 128ms/step - accuracy: 0.1034 - loss: 5.7506 - val_accuracy: 0.0998 - val_loss: 6.2867
Epoch 4/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 80s 133ms/step - accuracy: 0.1180 - loss: 5.4626 - val_accuracy: 0.1033 - val_loss: 6.3401
Epoch 5/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 79s 132ms/step - accuracy: 0.1306 - loss: 5.2033 - val_accuracy: 0.1064 - val_loss: 6.3658
Epoch 6/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 80s 134ms/step - accuracy: 0.1429 - loss: 4.9656 - val_accuracy: 0.1081 - val_loss: 6.4238
Epoch 7/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 82s 137ms/step - accuracy: 0.1561 - loss: 4.7446 - val_accuracy: 0.1105 - val_loss: 6.5207
Epoch 8/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 78s 130ms/step - accuracy: 0.1731 -